# 01. Canonical Cloudless DET Feedback / Local Qwen Benchmark

이 노트북은 `version0_15_update20260413/scripts`를 직접 실행하지 않습니다.  
최종 실행 경로는 canonical runtime입니다.

```text
utils/ga_search/model_suite_benchmark.py
utils/ga_search/local_model_registry.py
utils/ga_search/local_worker.py
utils/ga_search/candidate_generation.py
utils/ga_search/evaluation.py
```

중요 구분:

- `llm_mode=mock`: evaluator sanity check 전용. GT를 Generated로 복사할 수 있으므로 모델 성능이 아닙니다.
- `llm_mode=worker`: 실제 local Qwen 생성 경로입니다.
- 먼저 `qwen25_coder_7b`로 smoke를 통과시킨 뒤 14B를 실행합니다.
- `max_new_tokens=64`는 JSON이 잘려 `invalid_json`이 날 수 있으므로, 7B smoke 기본값은 `512`입니다.

현재 서버 이슈 대응:

- `torch.ones(..., device="cuda")`가 성공해야 모델 실행을 시작합니다.
- `ollama serve`가 `/dev/nvidia-uvm`을 잡고 있으면 PyTorch CUDA 초기화가 실패할 수 있습니다.
- 필요 시 터미널에서 `sudo systemctl stop ollama` 후 CUDA test를 다시 수행하세요.

In [ ]:
# ============================================================
# Setup
# ============================================================

import os
import sys
import json
import time
import shlex
import subprocess
from pathlib import Path
from datetime import datetime

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 260)
pd.set_option("display.max_colwidth", 300)

BASE_DIR = Path(os.environ.get("JOILANG_BASE_DIR", "/home/mgjeong/Desktop/llm/JOILang-Server")).expanduser().resolve()
assert BASE_DIR.exists(), f"BASE_DIR not found: {BASE_DIR}"

# Use the CUDA-working env created for this task.
# If your env name/path differs, override JOI_PY before running the notebook.
JOI_PY = Path(os.environ.get("JOI_PY", "/home/mgjeong/miniconda3/envs/joi/bin/python")).expanduser().resolve()
assert JOI_PY.exists(), f"JOI_PY not found: {JOI_PY}"

DATASET = BASE_DIR / "datasets" / "JOICommands-280.csv"
SERVICE_SCHEMA = BASE_DIR / "datasets" / "service_list_ver2.0.1.json"
LOCAL_MODEL_BASE = Path(os.environ.get("LOCAL_MODEL_BASE", "/home/mgjeong/Desktop/llm/local_models")).expanduser().resolve()

MODEL = os.environ.get("JOI_GA_MODEL", "gpt_mg.version0_13")
ROW_NO = int(os.environ.get("ROW_NO", "251"))

# Start with 7B. 14B is slower and should be run only after 7B succeeds.
MODEL_KEY_7B = os.environ.get("MODEL_KEY_7B", "qwen25_coder_7b")
MODEL_KEY_14B = os.environ.get("MODEL_KEY_14B", "qwen25_coder_14b")

CUDA_VISIBLE_DEVICES = os.environ.get("CUDA_VISIBLE_DEVICES_FOR_NOTEBOOK", "0")
LOCAL_DEVICE = os.environ.get("LOCAL_DEVICE", "cuda:0")

# 64 was proven too short and produced invalid_json. Use 512 for row-level smoke.
MAX_NEW_TOKENS_7B = int(os.environ.get("MAX_NEW_TOKENS_7B", "512"))
MAX_NEW_TOKENS_14B = int(os.environ.get("MAX_NEW_TOKENS_14B", "512"))

RUN_TAG = os.environ.get("RUN_TAG", datetime.now().strftime("%Y%m%d_%H%M%S"))
NB_ROOT = BASE_DIR / "artifacts" / "ga_search_tutorial_runs" / f"canonical_local_{RUN_TAG}"
NB_ROOT.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("JOI_PY:", JOI_PY)
print("LOCAL_MODEL_BASE:", LOCAL_MODEL_BASE)
print("NB_ROOT:", NB_ROOT)
print("MODEL:", MODEL)
print("ROW_NO:", ROW_NO)
print("CUDA_VISIBLE_DEVICES:", CUDA_VISIBLE_DEVICES)

In [ ]:
# ============================================================
# Helpers
# ============================================================

def ts():
    return datetime.now().strftime("%Y%m%d_%H%M%S")

def out_dir(label: str) -> Path:
    p = NB_ROOT / label
    p.mkdir(parents=True, exist_ok=True)
    return p

def run_cmd(cmd, *, log_path: Path | None = None, check: bool = False, timeout_sec: int | None = None, env_extra: dict | None = None):
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["CUDA_VISIBLE_DEVICES"] = CUDA_VISIBLE_DEVICES
    # Avoid stale /usr/local/cuda-* LD paths interfering with PyTorch wheel CUDA runtime.
    env["LD_LIBRARY_PATH"] = ""
    if env_extra:
        env.update({k: str(v) for k, v in env_extra.items()})

    print("\n[CMD]")
    print(" ".join(shlex.quote(str(x)) for x in cmd))

    started = time.time()
    proc = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(BASE_DIR),
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        timeout=timeout_sec,
    )
    elapsed = time.time() - started
    output = proc.stdout or ""

    if log_path:
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log_path.write_text(output, encoding="utf-8", errors="replace")
        print("[LOG]", log_path)

    print(output[-6000:])
    print(f"[RC] {proc.returncode}  [elapsed] {elapsed:.2f}s")

    if check and proc.returncode != 0:
        raise RuntimeError(f"command failed rc={proc.returncode}: {' '.join(map(str, cmd))}")

    return proc.returncode, output

def load_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))

def read_csv_or_empty(path: Path) -> pd.DataFrame:
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    return pd.read_csv(path)

def suite_model_dir(root: Path, model_key: str) -> Path:
    return root / model_key

def suite_summary(root: Path):
    p = root / "suite_summary.json"
    return load_json(p) if p.exists() else None

def model_summary(root: Path, model_key: str):
    p = suite_model_dir(root, model_key) / "model_summary.json"
    return load_json(p) if p.exists() else None

def candidates_df(root: Path, model_key: str) -> pd.DataFrame:
    return read_csv_or_empty(suite_model_dir(root, model_key) / "candidates" / "generation_000.csv")

def eval_df(root: Path, model_key: str) -> pd.DataFrame:
    return read_csv_or_empty(suite_model_dir(root, model_key) / "eval" / "row_evaluation.csv")

def show_file(path: Path, max_chars: int = 8000):
    print(f"\n===== {path} =====")
    if not path.exists():
        print("MISSING")
        return
    text = path.read_text(encoding="utf-8", errors="replace")
    print(text[:max_chars])
    if len(text) > max_chars:
        print(f"\n... truncated {len(text)-max_chars} chars ...")

def inspect_run(root: Path, model_key: str):
    print("root:", root)
    print("model_summary:")
    ms = model_summary(root, model_key)
    print(json.dumps(ms or {}, ensure_ascii=False, indent=2))

    cdf = candidates_df(root, model_key)
    edf = eval_df(root, model_key)

    print("\nCandidates:")
    if cdf.empty:
        print("No candidates.")
    else:
        cols = [c for c in [
            "row_no", "candidate_strategy", "backend",
            "generation_error_type", "generation_error_count",
            "raw_response_path", "generated_code"
        ] if c in cdf.columns]
        display(cdf[cols].head(20))

    print("\nEval:")
    if edf.empty:
        print("No eval rows.")
    else:
        cols = [c for c in [
            "row_no", "category", "det_score", "det_pass",
            "failure_reasons", "generated_code", "gt_code", "raw_response_path"
        ] if c in edf.columns]
        display(edf[cols].head(20))

    if not cdf.empty and "raw_response_path" in cdf.columns:
        raw = Path(str(cdf.iloc[0]["raw_response_path"]))
        if not raw.is_absolute():
            raw = BASE_DIR / raw
        show_file(raw, max_chars=12000)

## 1. CUDA readiness check

이 셀이 실패하면 모델 benchmark를 실행하지 마세요.  
`torch.ones(1, device="cuda")`가 성공해야 합니다.

In [ ]:
cuda_check_cmd = [
    str(JOI_PY), "-c",
    (
        "import torch, os; "
        "print('CUDA_VISIBLE_DEVICES:', os.environ.get('CUDA_VISIBLE_DEVICES')); "
        "print('torch:', torch.__version__); "
        "print('torch cuda:', torch.version.cuda); "
        "print('is_available:', torch.cuda.is_available()); "
        "print('device_count:', torch.cuda.device_count()); "
        "x=torch.ones(1, device='cuda'); "
        "print(x, x.device); "
        "print(torch.cuda.get_device_name(0))"
    )
]
rc, _ = run_cmd(cuda_check_cmd, log_path=out_dir("cuda_check") / "cuda_check.log", check=False, timeout_sec=120)
if rc != 0:
    raise RuntimeError(
        "CUDA check failed. Stop GPU-holding services such as ollama, reload/reboot NVIDIA runtime, "
        "then rerun this cell before benchmark."
    )

## 2. Canonical import / compile / render smoke

`version0_15_update20260413` 경로를 import하지 않는지 확인합니다.

In [ ]:
import_check = r'''
import importlib.util
for name in [
    "utils.ga_search.cli",
    "utils.ga_search.model_resolver",
    "utils.ga_search.render_adapter",
    "utils.ga_search.candidate_generation",
    "utils.ga_search.evaluation",
    "utils.ga_search.ga_engine",
    "utils.ga_search.local_model_registry",
    "utils.ga_search.model_suite_benchmark",
    "utils.det_evaluator",
]:
    spec = importlib.util.find_spec(name)
    print(name, "=>", spec.origin if spec else None)
    assert spec is not None
    assert "version0_15_update20260413" not in str(spec.origin)
'''
run_cmd([str(JOI_PY), "-c", import_check], log_path=out_dir("import_origin_check") / "import_origin_check.log", check=True)
run_cmd([str(JOI_PY), "-m", "compileall", "utils/ga_search", "utils/det_evaluator.py"], log_path=out_dir("compileall") / "compileall.log", check=True)
run_cmd([
    str(JOI_PY), "-m", "utils.ga_search.cli", "render",
    "--model", MODEL,
    "--user-input", "Turn on the light.",
    "--search-mode", "auto",
    "--dry-run",
], log_path=out_dir("render_smoke") / "render_smoke.log", check=True)

## 3. Local model preflight: Qwen 7B

모델 경로가 실제 local model/snapshot으로 resolve되는지 확인합니다.

In [ ]:
preflight_7b_dir = out_dir("preflight_qwen7b")
run_cmd([
    str(JOI_PY), "-m", "utils.ga_search.model_suite_benchmark",
    "--model-key", MODEL_KEY_7B,
    "--local-model-base-dir", str(LOCAL_MODEL_BASE),
    "--worker-python", str(JOI_PY),
    "--preflight-only",
    "--output-dir", str(preflight_7b_dir),
], log_path=preflight_7b_dir / "preflight.log", check=True, timeout_sec=120)

preflight_csv = preflight_7b_dir / "preflight.csv"
display(read_csv_or_empty(preflight_csv))

## 4. Real Qwen 7B single-row worker benchmark

실제 local model generation입니다. GT copy가 아닙니다.

- 기본 row: 251
- 기본 max_new_tokens: 512
- `max_new_tokens=64`는 JSON이 중간에서 잘려 `invalid_json`이 났으므로 사용하지 않습니다.
- 첫 목표는 DET pass가 아니라 `generation_error_rate=0.0`, prompt/completion token > 0, raw response 생성입니다.

In [ ]:
qwen7b_root = out_dir(f"qwen7b_row{ROW_NO}_{ts()}")

cmd = [
    str(JOI_PY), "-m", "utils.ga_search.model_suite_benchmark",
    "--model", MODEL,
    "--model-key", MODEL_KEY_7B,
    "--row-no", str(ROW_NO),
    "--llm-mode", "worker",
    "--local-model-base-dir", str(LOCAL_MODEL_BASE),
    "--worker-python", str(JOI_PY),
    "--local-device", LOCAL_DEVICE,
    "--local-files-only", "true",
    "--local-trust-remote-code", "true",
    "--local-max-new-tokens", str(MAX_NEW_TOKENS_7B),
    "--timeout-sec", "900",
    "--output-dir", str(qwen7b_root),
]
rc, _ = run_cmd(cmd, log_path=qwen7b_root / "run.log", check=False, timeout_sec=1200)
print("qwen7b_rc:", rc)

inspect_run(qwen7b_root, MODEL_KEY_7B)

ms = model_summary(qwen7b_root, MODEL_KEY_7B) or {}
if float(ms.get("generation_error_rate", 1.0) or 1.0) != 0.0:
    print("\n[WARN] generation_error_rate != 0. Check raw response above.")
else:
    print("\n[OK] Real local Qwen 7B generation completed without generation error.")

## 5. Optional: retry Qwen 7B with longer output budget

위 셀이 `invalid_json`이면 모델이 JSON을 만들다가 잘린 것입니다.  
아래를 켜서 `max_new_tokens=1024`로 재시도하세요.

In [ ]:
RUN_QWEN7B_LONG_RETRY = False

if RUN_QWEN7B_LONG_RETRY:
    qwen7b_long_root = out_dir(f"qwen7b_row{ROW_NO}_long_{ts()}")
    cmd = [
        str(JOI_PY), "-m", "utils.ga_search.model_suite_benchmark",
        "--model", MODEL,
        "--model-key", MODEL_KEY_7B,
        "--row-no", str(ROW_NO),
        "--llm-mode", "worker",
        "--local-model-base-dir", str(LOCAL_MODEL_BASE),
        "--worker-python", str(JOI_PY),
        "--local-device", LOCAL_DEVICE,
        "--local-files-only", "true",
        "--local-trust-remote-code", "true",
        "--local-max-new-tokens", "1024",
        "--timeout-sec", "1200",
        "--output-dir", str(qwen7b_long_root),
    ]
    rc, _ = run_cmd(cmd, log_path=qwen7b_long_root / "run.log", check=False, timeout_sec=1500)
    print("qwen7b_long_rc:", rc)
    inspect_run(qwen7b_long_root, MODEL_KEY_7B)
else:
    print("Long retry skipped. Set RUN_QWEN7B_LONG_RETRY=True only if 512-token run still returns invalid_json.")

## 6. Optional: Qwen 14B

14B는 느리고 VRAM을 더 많이 사용합니다. 7B generation이 성공한 뒤에만 실행하세요.

In [ ]:
RUN_QWEN14B = False

if RUN_QWEN14B:
    qwen14b_root = out_dir(f"qwen14b_row{ROW_NO}_{ts()}")
    cmd = [
        str(JOI_PY), "-m", "utils.ga_search.model_suite_benchmark",
        "--model", MODEL,
        "--model-key", MODEL_KEY_14B,
        "--row-no", str(ROW_NO),
        "--llm-mode", "worker",
        "--local-model-base-dir", str(LOCAL_MODEL_BASE),
        "--worker-python", str(JOI_PY),
        "--local-device", LOCAL_DEVICE,
        "--local-files-only", "true",
        "--local-trust-remote-code", "true",
        "--local-max-new-tokens", str(MAX_NEW_TOKENS_14B),
        "--timeout-sec", "1800",
        "--output-dir", str(qwen14b_root),
    ]
    rc, _ = run_cmd(cmd, log_path=qwen14b_root / "run.log", check=False, timeout_sec=2200)
    print("qwen14b_rc:", rc)
    inspect_run(qwen14b_root, MODEL_KEY_14B)
else:
    print("Qwen 14B skipped. Set RUN_QWEN14B=True after Qwen 7B succeeds.")

## 7. Optional mock foundation check

이 셀은 모델 평가가 아닙니다. evaluator sanity check 용도이며, GT copy가 발생할 수 있습니다.

In [ ]:
RUN_MOCK_FOUNDATION = False

if RUN_MOCK_FOUNDATION:
    mock_root = out_dir(f"mock_row{ROW_NO}_{ts()}")
    cmd = [
        str(JOI_PY), "-m", "utils.ga_search.cli", "eval",
        "--model", MODEL,
        "--dataset", str(DATASET),
        "--service-schema", str(SERVICE_SCHEMA),
        "--llm-mode", "mock",
        "--engine-mode", "mock",
        "--model-key", MODEL_KEY_7B,
        "--det-profile", "strict",
        "--det-threshold", "70",
        "--out-dir", str(mock_root),
        "--print-mode", "summary",
        "--row-no", str(ROW_NO),
    ]
    rc, _ = run_cmd(cmd, log_path=mock_root / "mock_eval.log", check=False, timeout_sec=300)
    print("mock_rc:", rc)
    # This path uses the regular eval layout, not suite layout.
    summary_path = mock_root / "eval" / "summary.json"
    if summary_path.exists():
        print(json.dumps(load_json(summary_path), ensure_ascii=False, indent=2))
else:
    print("Mock foundation check skipped.")